# SOV Sovereign AI — Training Data Generator

This notebook generates training data from the **12 Sovereign Pillars** using a local
LLM (sov33-master-v2 or qwen2.5:0.5b fallback) installed via Ollama.

The output is a JSONL file suitable for fine-tuning or continued pre-training of
sovereign-aligned models.  Each line is a prompt–completion pair rooted in one of the
12 pillars.

---
**Context:** SOV (Sovereign Operating Vehicle) — a framework for building AI that is
offline-first, human-centric, cryptographically auditable, and free from corporate
or government control.

**The 12 Sovereign Pillars:**
 1. Human-Centric Design
 2. Offline-First Architecture
 3. Cryptographic Attestation
 4. Transparent Provenance
 5. Federated Consensus
 6. Privacy by Default
 7. Deterministic Verification
 8. Open Standards
 9. Graceful Degradation
10. Economic Sovereignty
11. Interoperable Identity
12. Continuous Audit
---

In [ ]:
# Install Ollama
import json, time, hashlib, os, re

!curl -fsSL https://ollama.com/install.sh | sh
print("[SOV] Ollama installed")

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')
print("[SOV] Google Drive mounted")

In [ ]:
# Pull the SOV model; fall back to qwen2.5:0.5b if unavailable
MODEL = "sov33-master-v2:latest"

import subprocess
result = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True, timeout=120)
if result.returncode != 0:
    print(f"[SOV] {MODEL} not found, falling back to qwen2.5:0.5b")
    MODEL = "qwen2.5:0.5b"
    !ollama pull {MODEL}
else:
    print(f"[SOV] {MODEL} pulled successfully")

print(f"[SOV] Active model: {MODEL}")

In [ ]:
# ---------------------------------------------------------------------------
#  12 Sovereign Pillars — Training Prompt Templates
# ---------------------------------------------------------------------------

PILLARS = [
    {
        "id": 1,
        "name": "Human-Centric Design",
        "prompts": [
            "Explain why human-centric design is the foundation of sovereign AI.",
            "How does human-centric AI differ from profit-centric AI? Give three examples.",
            "Describe a scenario where human-centric design prevents algorithmic harm.",
            "What metrics should be used to evaluate human-centric AI systems?",
        ]
    },
    {
        "id": 2,
        "name": "Offline-First Architecture",
        "prompts": [
            "Why must sovereign AI operate offline-first? Discuss network independence.",
            "Design a minimal offline-first architecture for a peer-to-peer AI node.",
            "What are the security benefits of an offline-first approach to AI inference?",
            "How does offline-first architecture protect against surveillance capitalism?",
        ]
    },
    {
        "id": 3,
        "name": "Cryptographic Attestation",
        "prompts": [
            "Explain the role of cryptographic attestation in a self-verifying AI system.",
            "How would you implement a Merkle-tree-based attestation chain for model outputs?",
            "Compare digital signatures vs zero-knowledge proofs for AI attestation.",
            "What happens to the attestation chain when a model is updated?",
        ]
    },
    {
        "id": 4,
        "name": "Transparent Provenance",
        "prompts": [
            "Define transparent provenance in the context of AI decision-making.",
            "How can provenance graphs be used to audit an AI's reasoning chain?",
            "What metadata should every AI-generated output carry for full provenance?",
            "Explain why black-box AI is incompatible with transparent provenance.",
        ]
    },
    {
        "id": 5,
        "name": "Federated Consensus",
        "prompts": [
            "Describe how federated consensus works across sovereign AI nodes.",
            "What consensus algorithm is most appropriate for a trustless AI network?",
            "How does federated consensus prevent a single point of failure in AI governance?",
            "Compare federated consensus with centralized model governance.",
        ]
    },
    {
        "id": 6,
        "name": "Privacy by Default",
        "prompts": [
            "What does 'privacy by default' mean for an AI system?",
            "Design a data-handling policy that ensures privacy by default in federated learning.",
            "How does differential privacy apply to sovereign AI training?",
            "Why is privacy by default a sovereign right, not a feature?",
        ]
    },
    {
        "id": 7,
        "name": "Deterministic Verification",
        "prompts": [
            "Explain the importance of deterministic verification for AI safety.",
            "How can a deterministic verification layer be added to a probabilistic model?",
            "What role do formal methods play in deterministic AI verification?",
            "Contrast deterministic verification with statistical testing for high-stakes AI.",
        ]
    },
    {
        "id": 8,
        "name": "Open Standards",
        "prompts": [
            "Why must sovereign AI be built on open standards?",
            "List three open standards that are critical for interoperable AI systems.",
            "How do open standards prevent vendor lock-in for AI infrastructure?",
            "What is the role of IETF/W3C-style standardization in sovereign AI?",
        ]
    },
    {
        "id": 9,
        "name": "Graceful Degradation",
        "prompts": [
            "Define graceful degradation in the context of a distributed AI system.",
            "Describe a fallback strategy when a sovereign AI node loses network connectivity.",
            "How does graceful degradation improve the resilience of peer-to-peer AI?",
            "What is the opposite of graceful degradation, and why is it dangerous?",
        ]
    },
    {
        "id": 10,
        "name": "Economic Sovereignty",
        "prompts": [
            "Explain the concept of economic sovereignty in AI.",
            "How can token-based microeconomies fund sovereign AI infrastructure?",
            "What prevents a sovereign AI system from being captured by venture capital?",
            "Design a sustainable funding model for a community-owned AI node.",
        ]
    },
    {
        "id": 11,
        "name": "Interoperable Identity",
        "prompts": [
            "What is interoperable identity and why does sovereign AI need it?",
            "How do DIDs (Decentralized Identifiers) enable cross-platform AI identity?",
            "Describe an identity layer that allows AI agents to authenticate across sovereign nodes.",
            "What threat model applies to identity in a fully sovereign AI network?",
        ]
    },
    {
        "id": 12,
        "name": "Continuous Audit",
        "prompts": [
            "Why must sovereign AI be subject to continuous, not periodic, audit?",
            "Design an audit log format that captures every inference, decision, and attestation.",
            "How can audit data be stored immutably without a centralized ledger?",
            "What role do watchdogs and validators play in a continuously audited AI system?",
        ]
    },
]

total_prompts = sum(len(p["prompts"]) for p in PILLARS)
print(f"[SOV] Loaded {len(PILLARS)} pillars with {total_prompts} training prompts")

In [ ]:
# ---------------------------------------------------------------------------
#  Generate training data — one JSONL entry per prompt
# ---------------------------------------------------------------------------
import urllib.request

def generate(model: str, prompt: str, max_seconds: int = 60) -> str:
    """Send a prompt to Ollama and return the completion."""
    payload = json.dumps({
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"num_predict": 512, "temperature": 0.7}
    }).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"}
    )
    try:
        with urllib.request.urlopen(req, timeout=max_seconds) as resp:
            return json.loads(resp.read().decode()).get("response", "")
    except Exception as e:
        return f"[GENERATION ERROR] {e}"


training_records = []

for pillar in PILLARS:
    sys_prompt = f"You are an expert on the Sovereign AI pillar: {pillar['name']}. Provide detailed, technically precise answers."
    for i, user_prompt in enumerate(pillar["prompts"]):
        print(f"  Generating {pillar['id']}.{i+1}: {user_prompt[:60]}...")
        completion = generate(MODEL, f"{sys_prompt}\n\nUser: {user_prompt}\n\nAssistant:")
        record = {
            "pillar_id": pillar["id"],
            "pillar_name": pillar["name"],
            "prompt": user_prompt,
            "completion": completion,
            "model": MODEL,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "source": "sov_pillar_training_generator"
        }
        training_records.append(record)
        time.sleep(0.5)

print(f"\n[SOV] Generated {len(training_records)} training records")

In [ ]:
# ---------------------------------------------------------------------------
#  Save as JSONL to Google Drive and local Colab storage
# ---------------------------------------------------------------------------

# Local path (always available)
local_path = "/content/sov_pillar_training.jsonl"
with open(local_path, "w") as f:
    for rec in training_records:
        f.write(json.dumps(rec, sort_keys=True, ensure_ascii=False) + "\n")
print(f"[SOV] Saved locally: {local_path}")
print(f"[SOV] File size: {os.path.getsize(local_path):,} bytes")

# Google Drive path (persistent)
drive_path = "/content/drive/MyDrive/sov_pillar_training.jsonl"
try:
    with open(drive_path, "w") as f:
        for rec in training_records:
            f.write(json.dumps(rec, sort_keys=True, ensure_ascii=False) + "\n")
    print(f"[SOV] Saved to Drive: {drive_path}")
    print(f"[SOV] Drive file size: {os.path.getsize(drive_path):,} bytes")
except Exception as e:
    print(f"[SOV] Could not save to Drive (may not be mounted): {e}")

# Print first 3 records as preview
print("\n" + "=" * 64)
print("FIRST 3 TRAINING RECORDS (preview)")
print("=" * 64)
for rec in training_records[:3]:
    print(json.dumps(rec, indent=2, ensure_ascii=False))
    print("---")

# SHA-256 checksum of full dataset
with open(local_path, "rb") as f:
    digest = hashlib.sha256(f.read()).hexdigest()
print(f"\n[SOV] SHA-256 of training data: {digest}")
print("[SOV] Done — training data ready for fine-tuning")

---
**SOV Training Data Generator — Colab Edition**

The generated JSONL file contains prompt–completion pairs for all 12 Sovereign
Pillars.  This data can be used for:

- **Supervised fine-tuning (SFT)** of open-weight models to align with sovereign principles.
- **Continued pre-training** to inject sovereign AI concepts into the model's latent space.
- **Evaluation** of model adherence to the 12 pillars.

**File locations:**
- Colab VM: `/content/sov_pillar_training.jsonl`
- Google Drive: `/content/drive/MyDrive/sov_pillar_training.jsonl`

*"A sovereign AI is built on sovereign data." — SOV Protocol*